In [1]:
# Workflow Agents in Google ADK: Itinerary-Driven Packing List Example
# Welcome! In this notebook, you'll learn how to use Workflow Agents in Google's Agent Development Kit (ADK)
# to chain together two LLM agents: one that creates a travel itinerary, and another that generates a packing list
# based on that itinerary. This mirrors how real travelers plan: first decide what you'll do, then pack accordingly!

In [2]:
# Install Google ADK for Python
%pip install google-adk --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# (optional) Verify installation
%pip show google-adk

Name: google-adk
Version: 1.0.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: authlib, click, fastapi, google-api-python-client, google-cloud-aiplatform, google-cloud-secret-manager, google-cloud-speech, google-cloud-storage, google-genai, graphviz, mcp, opentelemetry-api, opentelemetry-exporter-gcp-trace, opentelemetry-sdk, pydantic, python-dotenv, PyYAML, sqlalchemy, tzlocal, uvicorn
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Configure environment
import os
# We use Gemini as our language model and ensure we are not using Vertex AI for this demo.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

In [5]:
# Import Required Modules
from google.adk.agents import Agent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types
from IPython.display import Markdown, display

In [6]:
# Itinerary Agent: Definition
# This agent creates a detailed itinerary using Google Search for up-to-date recommendations.
itinerary_agent = Agent(
    name="itinerary_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="An agent that creates a travel itinerary for a given destination and trip duration, using Google Search for up-to-date recommendations.",
    instruction="""
        You are a helpful and creative travel itinerary planner.
        When the user provides a destination and trip duration, use the [google_search] tool to find current and popular attractions, restaurants, and activities for each day.
        For each day, create a detailed schedule with 2-4 activities, including at least one meal suggestion and one local attraction, prioritizing the user's interests (such as art and food).
        Present the itinerary in a clear, easy-to-read markdown format, organized by day.
        If you need more information, ask the user for preferences or interests, but if you have enough, proceed to generate a full itinerary.
        Example format:

        ### Day 1
        - Morning: [Attraction or activity]
        - Lunch: [Restaurant or food experience]
        - Afternoon: [Attraction or activity]
        - Evening: [Dinner suggestion or event]

        ### Day 2
        ...

        Always use the [google_search] tool to find the latest recommendations for each activity or restaurant.
        Be concise, friendly, and ensure the itinerary is complete for all days requested.
    """,
    tools=[google_search],
    output_key="itinerary"
)

In [7]:
# Packing List Agent: Definition
# This agent uses the itinerary to recommend exactly what to pack for the planned activities.
packing_list_agent = Agent(
    name="packing_list_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Generates a packing list based on itinerary, destination, and user preferences.",
    instruction="""
        You are a helpful travel assistant.
        The user will provide their destination, trip duration, and a detailed itinerary (including planned activities, locations, and types of events for each day).
        Your task is to generate a complete, practical packing list tailored to the specific activities and locations in the itinerary.
        Use the provided itinerary to determine what items are necessary for each day and activity.
        
        Itinerary: {itinerary}

        Carefully review the itinerary and consider what clothing, gear, or items are needed for each activity (for example: hiking boots for hiking, swimwear for beach days, formalwear for special dinners, museum-appropriate attire, etc.).
        Organize the packing list by category (such as Clothing, Toiletries, Electronics, Documents, and Other Essentials).
        If you notice a special activity or requirement in the itinerary, include specific items for it.
        If any information is missing, make reasonable assumptions based on the itinerary and destination—do not ask the user for more details unless absolutely necessary.
        Present the packing list in a clear, easy-to-read markdown format. For example:

        #### Clothing
        - Comfortable walking shoes
        - Lightweight jacket
        - Dress for dinner at a nice restaurant
        - Swimwear (if beach or pool in itinerary)
        - ...

        #### Toiletries
        - Toothbrush and toothpaste
        - Sunscreen
        - ...

        #### Electronics
        - Phone and charger
        - Plug adapter (if needed)
        - ...

        #### Documents
        - Passport
        - Travel insurance
        - ...

        #### Other Essentials
        - Daypack for city tours
        - Water bottle
        - ...

        Only ask clarifying questions if the itinerary is extremely vague or unclear. Otherwise, use your best judgment and generate a full, actionable packing list.
    """,
    output_key="packing_list"
)

In [8]:
# Sequential Agent: Defintion
# The packing list agent receives the itinerary output in its prompt via {itinerary}
travel_pipeline_agent = SequentialAgent(
    name="travel_pipeline_agent",
    sub_agents=[itinerary_agent, packing_list_agent],
    description="Plans a travel itinerary and prepares a packing list based on user preferences."
)

In [9]:
# Set up the session and user input
APP_NAME = "wanderwise_workflow_demo"
USER_ID = "user_001"
SESSION_ID = "workflow_session_001"
USER_INPUT = "I'm traveling to Tokyo for 3 days and I love art and food."

# Run the travel planning workflow
session_service = InMemorySessionService()
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

Session(id='workflow_session_001', app_name='wanderwise_workflow_demo', user_id='user_001', state={}, events=[], last_update_time=1748213242.543978)

In [10]:
# Run the full pipeline agent with user input
content = types.Content(role="user", parts=[types.Part(text=USER_INPUT)])
runner = Runner(
    agent=travel_pipeline_agent,
    app_name=APP_NAME,
    session_service=session_service
)
events = runner.run(
    user_id=USER_ID,
    session_id=SESSION_ID,
    new_message=content
)

# Parse and Display Outputs
for event in events:
    if event.is_final_response():
        # Concatenate all parts in the final response
        full_text = ''.join(part.text for part in event.content.parts if part.text)
        display(Markdown(f"{full_text}"))

Okay! Here is a possible itinerary for your 3-day trip to Tokyo, focusing on art and food:

Okay, based on your interests in art and food, here's a possible 3-day itinerary for your Tokyo trip:

### Day 1: Art Immersion in Ueno and Culinary Delights in Ginza

*   **Morning:** Visit the **Tokyo National Museum** in Ueno Park, Japan's oldest and largest museum, to dive into Japanese art and history. (See Refs 5, 10)
*   **Lunch:** Head to **Harajuku Gyoza Lou** for delicious and affordable gyoza. (See Ref 7)
*   **Afternoon:** Explore **Ueno Park**, enjoy the vibrant atmosphere, and visit one of the other museums like the **Tokyo Metropolitan Art Museum**. (See Ref 5)
*   **Evening:** Indulge in a culinary experience in Ginza, known for its high-end restaurants. Consider **Ginza Komon** or explore various dining options. (See Refs 4, 17)

### Day 2: Contemporary Art and Unique Dining Experiences

*   **Morning:** Immerse yourself in contemporary art at the **Mori Art Museum** in Roppongi, which offers stunning city views. (See Refs 2, 4, 15)
*   **Lunch:** Try **Gyukatsu Motomura** for a unique experience of grilling your own beef cutlet. (See Ref 6)
*   **Afternoon:** Explore the **teamLab Borderless** (relocating to Azabudai Hills) for a delirium-inducing digital art experience. (See Refs 5, 10, 12)
*   **Evening:** Enjoy a themed dining experience at the **Ninja Restaurant "NINJA Tokyo"** for entertainment with your meal. (See Refs 9, 11)

### Day 3: Culture and Food Exploration in Asakusa

*   **Morning:** Visit the historic **Senso-ji Temple** in Asakusa, followed by a stroll through **Nakamise-dori** for traditional crafts and snacks. (See Refs 5, 18)
*   **Lunch:** Enjoy a **Tsukiji Fish Market** food and culture walking tour. Sample seafood delicacies and experience the market's hustle and bustle. (See Refs 3, 5, 6)
*   **Afternoon:** Participate in a **sushi-making class** in Asakusa to learn the art of sushi.
*   **Evening:** Experience a **Kabuki** performance at the **Kabukiza Theater** for a rich cultural experience. (See Refs 5, 13)

I hope this itinerary will give you a great experience of Tokyo!


Okay, here's a packing list tailored for your 3-day art and food-focused trip to Tokyo:

#### Clothing
*   Comfortable walking shoes: Essential for exploring museums, parks, and markets.
*   Comfortable socks: Bring several pairs for all the walking.
*   Lightweight jacket or sweater: For potentially cooler evenings or indoor spaces.
*   Versatile pants or jeans: Suitable for museums, restaurants, and markets.
*   T-shirts or comfortable tops: Easy to layer and appropriate for daytime activities.
*   A nicer outfit: For dining at high-end restaurants in Ginza or the Kabuki performance.
*   Underwear: Pack enough for each day.

#### Toiletries
*   Toothbrush, toothpaste, floss
*   Shampoo, conditioner, soap
*   Deodorant
*   Sunscreen: Especially important for outdoor activities in Ueno Park and Asakusa.
*   Any personal medications
*   Hand sanitizer: Useful for navigating public spaces and food markets.
*   Wet wipes: For quick cleanups, especially after trying street food.

#### Electronics
*   Phone and charger
*   Portable power bank: To keep your phone charged during long days of exploring.
*   Plug adapter: Japan uses Type A and B plugs (100V).
*   Camera or phone with a good camera: For capturing art, food, and cultural experiences.

#### Documents
*   Passport
*   Copies of important documents: Keep them separate from the originals.
*   Japan Rail Pass (if applicable): If you plan on traveling outside of Tokyo.
*   Any tickets or reservations: For museums, Kabuki, sushi class, etc.
*   Travel insurance information

#### Other Essentials
*   Daypack: For carrying essentials during daily outings.
*   Reusable water bottle: To stay hydrated while exploring.
*   Small Japanese phrasebook or translation app: Helpful for communication.
*   Cash (Japanese Yen): While many places accept cards, cash is still useful, especially in smaller shops and markets.
*   Tissues or small towel: Useful for quick cleanups.
*   Small notebook and pen: For jotting down notes, restaurant recommendations, or sketching.
*   Reusable shopping bag: For purchases at markets and shops.


In [11]:
# 🎉 Congratulations!
# You’ve just built and run a SequentialAgent workflow in ADK that first creates an itinerary,
# then generates a packing list tailored to your actual plans.
# Try changing the user input or expanding the workflow with more steps!